## Notebook 01: Preparing Data for Machine Learning

In this notebook, we’ll load a classic dataset using `scikit-learn`, convert it to a Spark DataFrame, and persist it as a Delta table in Unity Catalog. This sets the stage for our distributed ML workflows that scale beyond your local machine.

Let’s get some data on the board!

In [ ]:
# Set up Databricks Connect session
from databricks.connect import DatabricksSession
import os
import config

# Pull connection values from config file
os.environ["DATABRICKS_HOST"] = config.DATABRICKS_HOST
os.environ["DATABRICKS_TOKEN"] = config.DATABRICKS_TOKEN
os.environ["DATABRICKS_CLUSTER_ID"] = config.DATABRICKS_CLUSTER_ID
os.environ["DATABRICKS_WORKSPACE_ID"] = config.DATABRICKS_WORKSPACE_ID

# Initialize Spark session routed through Databricks Connect
spark = DatabricksSession.builder.getOrCreate()


In [ ]:
# Define the table and model names to be used in Unity Catalog
CATALOG_NAME = "alexander_booth" 
SCHEMA_NAME = "default"
TABLE_NAME = "breast_cancer_training_data"

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

# Load the breast cancer dataset as a pandas DataFrame
data = load_breast_cancer(as_frame=True)

# Combine features and target into a single DataFrame, renaming columns for Spark compatibility
df_pd = pd.concat(
    [data.data.rename(columns=lambda x: x.replace(' ', '_')),  # Replace spaces with underscores
     data.target.rename("label")],                             # Rename target column to "label"
    axis=1
)

# Convert the pandas DataFrame to a Spark DataFrame
df = spark.createDataFrame(df_pd)

# Write the Spark DataFrame to Unity Catalog as a Delta table (overwrite mode)
df.write.mode("overwrite").saveAsTable(f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}")

In [ ]:
# Preview the loaded dataset
df.show(5)

---

### ➡️ Up Next: Notebook 02 – Training a Local Model on Remote Data

Now that our dataset is saved to Unity Catalog, we’ll show how to read it back using Databricks Connect — pulling data from the cloud while keeping model training local.

In this next notebook, we’ll train a logistic regression model using `scikit-learn`, powered by our familiar local Pandas stack but seamlessly integrated with distributed data access.

Let’s connect the dots between data at scale and fast local experimentation.
